# Phase 4 — the probe, and a search that only builds features

Everything before this notebook had the agent doing two jobs at once: inventing features
*and* choosing a model. This one takes the second job away, and the interesting part is
**why that is a measurement problem rather than a simplification**.

As always: you drive it by hand first, and the LLM only shows up at the very end.

---

## 1. The thing that was wrong

A run's leaderboard used to rank *experiments* — one model, its parameters, one table:

```
e9   catboost   t10   0.11171
e14  lgbm       t16   0.11255
```

Ask it "is t16 a better table than t10?" and it cannot answer. Something else moved.
It ranked them anyway, and the top row was what got packaged.

We counted what that cost, across all 16 runs in `_runs/`:

| | |
|---|---|
| experiments, all runs | **222** |
| varied only the model or its parameters | **97 (44%)** |
| runs whose winner was a model variation | **10 of 16** |

And the margins those winners were decided by, in each run's own `val_se`:
**0.03**, **0.05**, **0.18**. Three to eighteen *hundredths* of a standard error — coin
flips recorded as verdicts. On Porto Seguro one of those coin flips cost 0.012 of gini
on Kaggle, which was more than every feature the agent built after its second tool call.

The principle we landed on:

> **Search where the space is unbounded. Enumerate where it is small.**

Model types are a set of four. You never need to *search* a set you can simply train all
of. Feature space is unbounded and compositional, and is the only place a search loop
earns the noise it mines.

The full argument is in `docs/decisions/0035`, and `0036` corrects a claim it got wrong
— worth reading in that order, because the correction is the more interesting document.


## 2. A dataset we can reason about completely

`price` genuinely depends on `area`, `rooms` and `city`. `noise` is exactly what it says.
`city` is a string, so no model can consume it until something encodes it — that is the
one real feature-engineering decision available here, and we will watch the harness
notice it.

In [ ]:
import os
import tempfile

import numpy as np
import pandas as pd

rng = np.random.default_rng(0)
n = 900

df = pd.DataFrame({
    "area": rng.uniform(30, 200, n),
    "rooms": rng.integers(1, 7, n),
    "city": rng.choice(["berlin", "hamburg", "koeln", "essen"], n),
    "noise": rng.normal(size=n),
})
city_premium = {"berlin": 25_000, "hamburg": 8_000, "koeln": 0, "essen": -6_000}
df["price"] = (
    df["area"] * 900
    + df["rooms"] * 4_000
    + df["city"].map(city_premium)
    + rng.normal(0, 6_000, n)
    + 40_000
)

workdir = tempfile.mkdtemp(prefix="probe_demo_")
data_path = os.path.join(workdir, "houses.csv")
df.to_csv(data_path, index=False)
df.head()


## 3. Starting a run

`inspect_data` is still the first call and still resolves the task type and the metric.

Watch the third line. Nobody chose `log1p` — the metric is `rmsle`, which *is* squared
error on `log1p(y)`, so fitting on that scale makes the model's own loss the thing the
run is scored by. Decision 0021 originally made this an agent choice specifically so that
nobody would assume it; the assumption was then measured, it held, and 0022 supplies the
general reason. So it is derived now, and `TaskSpec.target_transform` overrides it for
the cases where nothing derives it (`mae` on a skewed target, say).

In [ ]:
from tabular_agent.harness.session import RunSession
from tabular_agent.harness import journal, tools
from tabular_agent.taskspec import TaskSpec

session = RunSession(os.path.join(workdir, "run"), data_path, TaskSpec(target="price"))
info = tools.inspect_data(session)

spec = session.resolved()
print("metric          :", spec.metric)
print("target_transform:", spec.target_transform, " <- derived from the metric, not chosen")
print("work rows       :", journal.load(session.run_dir)["split"]["n_work"])
print("val rows        :", journal.load(session.run_dir)["split"]["n_val"])


## 4. The ruler

A **probe** is not a candidate model. It is a measuring instrument: one estimator, fixed
parameters, fixed seed, applied unchanged to every table, so that a difference between
two tables is about the tables.

Two details that are load-bearing:

- **The seed is a module constant, not `TaskSpec.seed`.** A ruler that changes between
  runs cannot compare across runs, and comparing runs is most of what this project does.
- **The parameters are untuned, deliberately.** Tuning the ruler against a dataset would
  make the ruler a function of that dataset, which is the failure 0022 records.

In [ ]:
from tabular_agent.probe import describe_probe, default_target_transform

print(describe_probe())
for m in ["rmse", "rmsle", "rmspe", "mae"]:
    print(f"  {m:6s} -> {default_target_transform(m, 'regression')}")


## 5. One fit, not `k + 1`

Here is the piece that makes everything after it affordable.

The number tables are ranked by is the score on the **validation half** — the same rows
for every table, which is the whole basis of the comparison. That score comes from a
model fitted on *all* of the table's rows. So the number of folds never enters it.

One fit per table. Not `n_splits + 1`.

(Cross-validation has not gone away — it comes back in section 9, for a job that genuinely
needs it.)

In [ ]:
from tabular_agent.io import load_table
from tabular_agent.probe import fit_probe

work = load_table(session.data_path)
X = work.drop(columns=["price"]).select_dtypes(include="number")
y = work["price"]

fit = fit_probe(X, y, spec.task_type, spec.metric)
print("fitted on      :", list(X.columns))
print("target scale   :", fit.target_transform)
print("seconds        :", fit.fit_seconds)


Now score it on the held-back half. This is `probe_score`, and `probe_se` is its
standard error from a bootstrap over those rows.

In [ ]:
from tabular_agent.harness.tools import _score_on_val

score, se, why_not = _score_on_val(
    session, session.data_path, fit.model, fit.feature_names, spec, y,
    target_transform=fit.target_transform,
)
print(f"probe_score on the validation half: {score:.5f}  (+/- {se:.5f})")


## 6. Every table is scored as it is written

You no longer ask for a table to be trained on. It is measured on creation, and the
number comes back in the same reply.

The reason is **coverage**, and it came out of the run archive. Agents build chains —
t12, then t13, then t14 — and spend an experiment only on the endpoint. So across all 16
runs, of the 24 tables that recombined an abandoned branch, **22 could not be compared to
the branch they continued**, because that parent was never trained. The holes were
concentrated exactly where the interesting comparisons were.

`probe_vs_source_table` is the comparison that is actually a decision: this table against
the one it came from. Positive is better whatever the metric, like every other gain field.

Watch t1 pay for itself (`city` was unusable before) and t2 not (a ratio of two columns a
tree can already split on).

In [ ]:
t1 = tools.fit_encoding(session, ["city"], "target")
print("t1  target-encode city")
print(f"    probe_score          {t1['probe_score']:.5f}")
print(f"    probe_vs_source_table {t1['probe_vs_source_table']:+.5f}   (positive is always better)")

t2 = tools.transform_features(
    session,
    "def transform_features(df):\n"
    "    df = df.copy()\n"
    "    df['area_per_room'] = df['area'] / df['rooms']\n"
    "    return df",
    table=t1["table"],
)
print("t2  area_per_room, on top of t1")
print(f"    probe_score          {t2['probe_score']:.5f}")
print(f"    probe_vs_source_table {t2['probe_vs_source_table']:+.5f}")


## 7. The loophole that closed itself

Here is a table that adds a column carrying no information.

Under the old two-level stop rule, this was worth something: a new table reset the
per-table patience counter and bought the run more experiments. That loophole produced
e11 on Porto Seguro — **the best model this harness has ever made** (`research/archive/decisions/0034`,
footnote).

With one counter over tables there is nothing to reset. Look at the gain.

In [ ]:
t3 = tools.transform_features(
    session,
    "def transform_features(df):\n"
    "    df = df.copy()\n"
    "    df['constant'] = 1.0\n"
    "    return df",
    table=t2["table"],
)
print("t3  a column that carries nothing")
print(f"    probe_score          {t3['probe_score']:.5f}")
print(f"    probe_vs_source_table {t3['probe_vs_source_table']:+.5f}")
print()
print(t3["note"])


Every table carries its number, so `explore_journal`'s `tables` frame is now the run's
feature search, complete. `tables.sort_values("probe_score")` is the question the search
is actually asking.

In [ ]:
frames = tools._journal_frames(session)
frames["tables"][["id", "method", "probe_score", "probe_se", "probe_vs_source_table"]]


## 8. Why residuals cannot reuse the ranking fit

The agent still needs to know *where* its model is wrong — that is the only way a score
turns back into a hypothesis. The tempting shortcut is to reuse the fit we already paid
for.

Don't. Compare the two numbers below.

In [ ]:
from tabular_agent.cv import make_folds
from tabular_agent.metrics import compute_metric
from tabular_agent.models import train_cv
from tabular_agent.probe import PROBE_MODEL_TYPE, PROBE_PARAMS, PROBE_SEED

tbl = load_table(session.resolve_table(t2["table"]))
Xt = tbl.drop(columns=["price"]).select_dtypes(include="number")
yt = tbl["price"]

folds = make_folds(yt, spec.task_type, n_splits=spec.n_splits, seed=spec.seed)
cv = train_cv(Xt, yt, folds, spec.task_type, spec.metric,
              model_type=PROBE_MODEL_TYPE, params=dict(PROBE_PARAMS), seed=PROBE_SEED,
              target_transform=spec.target_transform)

in_sample = yt.to_numpy() - np.expm1(cv.final_model.predict(Xt))
out_of_fold = yt.to_numpy() - cv.oof

print(f"in-sample  residual, mean |error| : {np.abs(in_sample).mean():9.1f}")
print(f"out-of-fold residual, mean |error| : {np.abs(out_of_fold).mean():9.1f}")
print()
print("The first number is what the model failed to memorise.")
print("Only the second says anything about rows it has not seen.")


A boosted tree with 200 rounds fits its training rows closely; the residuals it leaves
behind describe what it failed to *memorise*. An agent grouping those by city would find
noise and build features against it.

So `explore_table` refits across folds, on request. `research/archive/decisions/0037` records why the
asymmetry — one fit to rank, `n_splits` fits to diagnose — is not an inconsistency waiting
to be tidied away.

One boundary worth stating: this reads the **work half only**. Residuals from the
validation half would be honest and would be the worst option on the list, because that is
the set tables are ranked on — an agent reading per-row errors there and writing features
in response is fitting features to the ranking set by hand.

The city premium we planted is exactly what shows up.

In [ ]:
out = tools.explore_table(
    session,
    table=t2["table"],
    code=(
        "def explore(df):\n"
        "    by_city = df.groupby('city')['residual'].agg(['mean', 'size'])\n"
        "    return by_city.round(0).to_dict('index')\n"
    ),
)
out["result"]


## 9. The terminal fit

The search is over. Only now does anything choose a model: `finalize_run` takes the best
table by `probe_score` and fits the enumerated set on it.

One argmax over three candidates — about `sqrt(2 ln 3)` ≈ 1.5 standard errors of luck —
in place of the 97 model-variation experiments it replaces.

It is deliberately **not** a blend. Averaging would mine no luck at all and is the better
answer in theory, but our own measurements disagree: Rossmann's OOF-weighted blend scored
0.12991 against 0.12741 for the best single model, and house-prices' three-family blend
bought +0.00024 — because boosted trees on one feature matrix have residual correlations
of 0.86–0.88. Every candidate stays on the leaderboard so that pick-best against
equal-weight is a one-line experiment the day we have an honest number to judge it with.

In [ ]:
final = tools.finalize_run(session)
lb = journal.load(session.run_dir)["leaderboard"]
pd.DataFrame([
    {"id": r["id"], "model": r["model_type"], "table": r["source_table"],
     "val_score": round(r["val_score"], 5)}
    for r in lb
])


In [ ]:
print("winner        :", final["best_experiment_id"], final["model_type"])
print("val_score     :", round(final["val_score"], 5))
print("baseline      :", round(final["baseline_score"], 5))
print("bundle        :", final["bundle_dir"])
print("unmeasured    :", final["unmeasured_tables"], " <- the attribution gap, now normally empty")


`unmeasured_tables` is the attribution gap, and it is empty. Every link in the winning
table's history was measured by the same ruler on the same rows, so the chain can be read
link by link. That used to require a deliberate extra experiment per link, and mostly did
not happen.

## 10. The claim that failed, and why it is here anyway

0035 argued that model-vs-model comparisons are much noisier than table-vs-table ones,
because different estimators make less correlated errors. It sounded right. It is wrong,
and the measurement is worth reproducing yourself.

Run the same rows through both candidates many times and look at the spread of the
*difference*:

```
Var(A - B) = Var(A) + Var(B) - 2 Cov(A, B)
```

On house-prices v8 the pairing benefit was 3–5x — and **identical** for model swaps and
table swaps (0.29–0.37 vs 0.18–0.30). Our own notes already said so: GBDT-vs-GBDT residual
correlation of 0.86–0.88 *is* the statement that swapping estimators leaves predictions
highly correlated.

What actually governs the covariance is how similar the two **predictors** are, which has
nothing to do with which knob you turned.

The probe survives that correction because it never depended on it — coverage, the removed
confound, and one fit per table are the reasons. `research/archive/decisions/0036`.

In [ ]:
def paired_bootstrap(preds_a, preds_b, y_true, metric, B=300, seed=0):
    """Resample the SAME validation rows for both candidates, many times."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    sa, sb = [], []
    for _ in range(B):
        idx = rng.integers(0, n, n)
        sa.append(compute_metric(metric, y_true[idx], preds_a[idx]))
        sb.append(compute_metric(metric, y_true[idx], preds_b[idx]))
    sa, sb = np.array(sa), np.array(sb)
    return {
        "sd(A)": sa.std(ddof=1),
        "sd(B)": sb.std(ddof=1),
        "sd(A-B)": (sa - sb).std(ddof=1),
        "if independent": np.sqrt(sa.var(ddof=1) + sb.var(ddof=1)),
        "corr": np.corrcoef(sa, sb)[0, 1],
    }


from tabular_agent.pipeline import Pipeline
from tabular_agent.probe import probe_predict
import json as _json

val = load_table(session.val_path)
y_val = val["price"].to_numpy()


def val_preds(table_rel):
    path = session.resolve_table(table_rel)
    tbl = load_table(path)
    Xf = tbl.drop(columns=["price"]).select_dtypes(include="number")
    f = fit_probe(Xf, tbl["price"], spec.task_type, spec.metric)
    lineage = _json.load(open(path.replace(".parquet", ".pipeline.json")))
    produced = Pipeline.from_dict(lineage).transform(val.drop(columns=["price"]))
    return probe_predict(f, produced[list(Xf.columns)], spec.task_type)


pa, pb = val_preds(t1["table"]), val_preds(t2["table"])
paired_bootstrap(pa, pb, y_val, spec.metric)


## 11. And only now, the LLM

The agent's whole menu. Notice what is missing: there is nothing here that chooses a
model, and that is the point rather than an omission.

Its job is columns — what to derive, what to encode, which rows to train on. The harness
does the rest, once, at the end.

---

### What this notebook does not settle

Every number in it is a `val_score` or a `probe_score`, and both are measured on rows the
search selects against. `research/archive/decisions/0034` shows what that costs: across five runs with
a Kaggle number, optimism ran +0.002 to +0.018 and grew with the number of experiments.

So the honest statement about everything above is that it is justified by the **structure**
of the comparisons, not by their values. Deciding whether it made the models better needs a
set nothing selected on. That is `research/archive/decisions/0023`, and it is built now — see
`notebooks/phase4_sealed_holdout.ipynb`, which measures the gap on rows this notebook's
search never touched.

In [ ]:
from tabular_agent.harness.agent import build_agent

agent = build_agent()
sorted(agent._function_toolset.tools)
